# Model Evaluation & Analysis

Deep-dive into model performance:
- Per-user analysis: which users benefit from which models
- Cold-start performance comparison
- Popularity bias analysis
- A/B test simulation
- Error analysis

In [ ]:
import sys
sys.path.insert(0, '..')

import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict

from src.data_loader import RetailRocketDataLoader
from src.evaluation import (
    precision_at_k, recall_at_k, ndcg_at_k, hit_rate_at_k,
    coverage, diversity, novelty
)
from src.utils import load_config

import os
PROJECT_ROOT = os.path.abspath('..')
config = load_config(os.path.join(PROJECT_ROOT, 'configs/config.yaml'))

%matplotlib inline
try:
    plt.style.use('seaborn-v0_8-whitegrid')
except OSError:
    plt.style.use('seaborn-whitegrid')

## 1. Load Models & Data

In [ ]:
loader = RetailRocketDataLoader()
loader.run_pipeline()
train, val, test = loader.temporal_train_test_split()

MODEL_DIR = os.path.join(PROJECT_ROOT, 'models')
models = {}
for name, filename in [('CF-KNN', 'collaborative_filtering.pkl'),
                        ('MF-SVD', 'matrix_factorization.pkl'),
                        ('Content-Based', 'content_based.pkl')]:
    path = os.path.join(MODEL_DIR, filename)
    if os.path.exists(path):
        with open(path, 'rb') as f:
            models[name] = pickle.load(f)
        print(f"Loaded {name}")

print(f"\nLoaded {len(models)} models")

## 2. Cold-Start Analysis

In [ ]:
train_user_counts = train.groupby('visitorid')['itemid'].nunique()

cold_users = set(train_user_counts[train_user_counts <= 5].index)
warm_users = set(train_user_counts[train_user_counts > 5].index)

test_gt = defaultdict(set)
for _, row in test.iterrows():
    test_gt[row['visitorid']].add(row['itemid'])

test_cold = {uid: items for uid, items in test_gt.items() if uid in cold_users}
test_warm = {uid: items for uid, items in test_gt.items() if uid in warm_users}

print(f"Cold-start users (≤5 train items): {len(test_cold)}")
print(f"Warm users (>5 train items): {len(test_warm)}")

# Compare model performance on cold vs warm users
cold_warm_results = {}
for model_name, model in models.items():
    for segment, users in [('Cold', test_cold), ('Warm', test_warm)]:
        ndcg_scores = []
        for uid, relevant in list(users.items())[:200]:
            try:
                if model_name == 'CF-KNN':
                    uidx = loader.user_to_idx.get(uid)
                    if uidx is None: continue
                    recs = model.predict_for_user(uidx, top_n=10)
                    rec_items = [loader.idx_to_item.get(idx, idx) for idx, _ in recs]
                else:
                    recs = model.predict_for_user(uid, top_n=10)
                    rec_items = [item_id for item_id, _ in recs]
                ndcg_scores.append(ndcg_at_k(rec_items, relevant, 10))
            except:
                continue
        cold_warm_results[f'{model_name} ({segment})'] = np.mean(ndcg_scores) if ndcg_scores else 0

fig, ax = plt.subplots(figsize=(12, 5))
names = list(cold_warm_results.keys())
values = list(cold_warm_results.values())
colors = ['#e74c3c' if 'Cold' in n else '#3498db' for n in names]
bars = ax.bar(names, values, color=colors)
ax.set_title('NDCG@10: Cold-Start vs Warm Users')
ax.tick_params(axis='x', rotation=45)
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005,
            f'{val:.3f}', ha='center', va='bottom')
plt.tight_layout()
plt.show()

## 3. Popularity Bias Analysis

In [ ]:
item_pop = train.groupby('itemid')['visitorid'].nunique().to_dict()
total_users = train['visitorid'].nunique()

pop_analysis = {}
for model_name, model in models.items():
    all_rec_pops = []
    sample_users = list(test_gt.keys())[:200]
    
    for uid in sample_users:
        try:
            if model_name == 'CF-KNN':
                uidx = loader.user_to_idx.get(uid)
                if uidx is None: continue
                recs = model.predict_for_user(uidx, top_n=10)
                rec_items = [loader.idx_to_item.get(idx, idx) for idx, _ in recs]
            else:
                recs = model.predict_for_user(uid, top_n=10)
                rec_items = [item_id for item_id, _ in recs]
            
            for item in rec_items:
                all_rec_pops.append(item_pop.get(item, 0))
        except:
            continue
    
    pop_analysis[model_name] = all_rec_pops

fig, ax = plt.subplots(figsize=(12, 5))
data = [pop_analysis[name] for name in pop_analysis if pop_analysis[name]]
labels = [name for name in pop_analysis if pop_analysis[name]]
ax.boxplot(data, labels=labels)
ax.set_ylabel('Item Popularity (# unique users in training)')
ax.set_title('Popularity Bias: Distribution of Recommended Item Popularity')
plt.tight_layout()
plt.show()

## 4. A/B Test Simulation

Simulate an A/B test comparing the hybrid model vs the best single model.

In [ ]:
from scipy import stats

model_a_name = 'MF-SVD'
model_b_name = 'CF-KNN'

model_a = models.get(model_a_name)
model_b = models.get(model_b_name)

if model_a and model_b:
    ndcg_a, ndcg_b = [], []
    sample_users = list(test_gt.keys())[:300]
    np.random.shuffle(sample_users)
    
    for uid in sample_users:
        relevant = test_gt[uid]
        try:
            recs_a = model_a.predict_for_user(uid, top_n=10)
            rec_items_a = [item_id for item_id, _ in recs_a]
            ndcg_a.append(ndcg_at_k(rec_items_a, relevant, 10))
        except:
            ndcg_a.append(0)
        
        try:
            uidx = loader.user_to_idx.get(uid)
            if uidx is not None:
                recs_b = model_b.predict_for_user(uidx, top_n=10)
                rec_items_b = [loader.idx_to_item.get(idx, idx) for idx, _ in recs_b]
                ndcg_b.append(ndcg_at_k(rec_items_b, relevant, 10))
            else:
                ndcg_b.append(0)
        except:
            ndcg_b.append(0)
    
    t_stat, p_value = stats.ttest_rel(ndcg_a, ndcg_b)
    
    print(f"=== A/B Test Simulation ===")
    print(f"Model A ({model_a_name}): Mean NDCG@10 = {np.mean(ndcg_a):.4f} ± {np.std(ndcg_a):.4f}")
    print(f"Model B ({model_b_name}): Mean NDCG@10 = {np.mean(ndcg_b):.4f} ± {np.std(ndcg_b):.4f}")
    print(f"Paired t-test: t={t_stat:.3f}, p={p_value:.4f}")
    print(f"Result: {'Statistically significant' if p_value < 0.05 else 'Not significant'} (α=0.05)")
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].hist(ndcg_a, bins=30, alpha=0.7, label=model_a_name, color='#3498db')
    axes[0].hist(ndcg_b, bins=30, alpha=0.7, label=model_b_name, color='#e74c3c')
    axes[0].set_title('NDCG@10 Distribution per User')
    axes[0].legend()
    
    diff = np.array(ndcg_a) - np.array(ndcg_b)
    axes[1].hist(diff, bins=30, color='#2ecc71', edgecolor='white')
    axes[1].axvline(x=0, color='red', linestyle='--')
    axes[1].set_title(f'Difference ({model_a_name} - {model_b_name})')
    axes[1].set_xlabel('NDCG@10 Difference')
    
    plt.tight_layout()
    plt.show()
else:
    print("Models not available for A/B test simulation")

## 5. Coverage & Diversity Summary

In [ ]:
n_total_items = len(loader.item_to_idx)
summary = []

for model_name, model in models.items():
    all_recs = []
    for uid in list(test_gt.keys())[:200]:
        try:
            if model_name == 'CF-KNN':
                uidx = loader.user_to_idx.get(uid)
                if uidx is None: continue
                recs = model.predict_for_user(uidx, top_n=10)
                rec_items = [loader.idx_to_item.get(idx, idx) for idx, _ in recs]
            else:
                recs = model.predict_for_user(uid, top_n=10)
                rec_items = [item_id for item_id, _ in recs]
            all_recs.append(rec_items)
        except:
            continue
    
    cov = coverage(all_recs, n_total_items)
    div = diversity(all_recs[:100])
    nov = np.mean([novelty(recs, item_pop, total_users) for recs in all_recs]) if all_recs else 0
    
    summary.append({
        'Model': model_name,
        'Coverage': f'{cov:.2%}',
        'Diversity': f'{div:.4f}',
        'Novelty': f'{nov:.2f}'
    })

pd.DataFrame(summary).set_index('Model')

## Key Findings

Document your analysis findings here after running:

1. **Best overall model**: [Model] with [NDCG@10] score
2. **Cold-start**: Content-based / hybrid handles cold users better than pure CF
3. **Popularity bias**: [Observations about which models over-recommend popular items]
4. **A/B test**: [Statistical significance of improvement]
5. **Coverage vs Accuracy trade-off**: [Observations]